<a href="https://colab.research.google.com/github/ArthurrCr/cloudband/blob/main/notebooks/00_baselines/score_ocm_cloudsen12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --quiet "omnicloudmask[legacy]==1.7.0" tacoreader transformers scipy

In [ ]:
REPO_URL = "https://github.com/ArthurrCr/cloudband.git"
PROJECT_DIR = "/content/cloudband"
BRANCH = "main"

import os
import sys

if not os.path.exists(PROJECT_DIR):
    !git clone --quiet {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git fetch --quiet origin && git reset --quiet --hard origin/{BRANCH}

SRC_DIR = f"{PROJECT_DIR}/src"
os.chdir(PROJECT_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

In [ ]:
import gc
import os
import re
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from cloudband.colab.session import reload_package, start

reload_package("cloudband")

from cloudband.baselines import ocm
from cloudband.datasets import cloudsen12 as dataset
from cloudband.datasets.cloudsen12 import read_sample as gdal_read_sample
from cloudband.datasets.fast_read import FastReader
from cloudband.eval import phase2 as evaluation
from cloudband.eval.palette import assign_colors
from cloudband.eval.plots import (
    plot_experiment_comparison,
    plot_paired_differences,
    plot_seed_boa,
    plot_unit_differences,
)
from cloudband.eval.report import compare, paper_style_table
from cloudband.models.ocm import PAPER_BACKBONES, build_unet
from cloudband.models.swin_upernet import build_swin_upernet
from cloudband.pipelines import phase0
from cloudband.pipelines.phase0 import select_rgn
from cloudband.train.cache import LocalCache
from cloudband.train.loop import checkpoint_name
from cloudband.train.protocol import PROTOCOL_REVISION, ocm_shared_protocol, swin_shared_protocol
from cloudband.train.store import RunStore, ensure_shared_folder, load_model_from_checkpoint

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

BASELINE_DIR = Path("/content/drive/MyDrive/cloudband/results")
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_FOLDER = Path("/content/drive/MyDrive/cloudband_treino")
ensure_shared_folder(DRIVE_FOLDER)
RESULTS_DIR = DRIVE_FOLDER / "results"
CHECKPOINT_DIR = DRIVE_FOLDER / "checkpoints"

session = start(PROJECT_DIR, require_accelerator=True)
store = RunStore(
    results_dir=RESULTS_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    source_dir=Path(PROJECT_DIR) / "models",
)
store.wake()

ocm_protocol = ocm_shared_protocol(learning_rate=1e-4)
swin_protocol = swin_shared_protocol(learning_rate=1e-4)
print(f"published OCM results: {BASELINE_DIR}")
print(f"phase 2 shared folder: {DRIVE_FOLDER}  |  runs: {ocm_protocol.run_id}, {swin_protocol.run_id}")

In [ ]:
import getpass
import os

import requests

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    token = getpass.getpass("Hugging Face token (type 'Read' token, hidden): ").strip()

os.environ.pop("HF_TOKEN", None)
if token:
    reply = requests.get(
        "https://huggingface.co/api/whoami-v2",
        headers={"Authorization": f"Bearer {token}"},
        timeout=30,
    )
    if reply.status_code == 200:
        os.environ["HF_TOKEN"] = token
        print(f"token accepted for the account {reply.json().get('name', '?')}")
    else:
        print(f"token refused (HTTP {reply.status_code}); going on without it")
else:
    print("no token: requests use the shared anonymous quota")
del token

In [ ]:
def load_test(remote_source):
    table = dataset.load_remote(remote_source)
    differences = dataset.verify_split_sizes(table)
    if differences:
        raise ValueError(f"split sizes differ from the documented ones: {differences}")
    return dataset.select(table, split="test")


test_table = load_test(dataset.REMOTE_L1C)
test_l2a_table = load_test(dataset.REMOTE_L2A)
print("test patches, L1C (primary):", len(test_table))
print("test patches, L2A (secondary):", len(test_l2a_table))
print("pairable:", dataset.expected_pairable_scenes(test_table))

## OmniCloudMask publicado, por versão do modelo

In [ ]:
ocm.check_version()

MODEL_VERSIONS = (ocm.PAPER_MODEL_VERSION, 2.0, 3.0, ocm.LATEST_MODEL_VERSION)
print("package:", ocm.package_version())
print("scoring model versions:", MODEL_VERSIONS)

In [ ]:
def report(position, total):
    if position % 25 == 0 or position == total:
        print(f"{position}/{total}", flush=True)

In [ ]:
LIMIT = None

def cached_csv(model_id: str, dataset_id: str, filename: str) -> pd.DataFrame | None:
    """Load a CSV from a previous run, or None if never saved there.

    Inference on OCM's published weights has no random component (fixed
    weights, pinned config), so a cached run is expected to match a fresh
    one; this only skips recomputation, it does not change what gets
    reported.
    """
    path = BASELINE_DIR / model_id / dataset_id / filename
    if not path.exists():
        return None
    return pd.read_csv(path, index_col="experiment")


results = {}
paper_tables = {}
for model_version in MODEL_VERSIONS:
    model_id = f"ocm-rgn-published-v{ocm.package_version()}-model{model_version}"

    scores = cached_csv(model_id, phase0.CLOUDSEN12_TEST, "scores.csv")
    paper_table = cached_csv(model_id, phase0.CLOUDSEN12_TEST, "paper_table.csv")
    if scores is not None and paper_table is not None:
        print(f"model_version={model_version} -> loaded from cache: {model_id}")
        results[model_version] = scores
        paper_tables[model_version] = paper_table
        continue

    config = ocm.InferenceConfig(
        patch_size=509, patch_overlap=0, model_version=model_version
    )
    print(f"model_version={model_version} -> running: {model_id}")

    result = phase0.run(
        test_table,
        lambda stack, config=config: ocm.predict_array(stack, config),
        model_id=model_id,
        limit=LIMIT,
        progress=report,
    )

    frame = phase0.compare_to_reference(result, phase0.STUPMASK_CLOUDSEN12)
    print("  pairable scenes:", result.pairable)
    print(" ", phase0.describe_comparison(frame))

    paths = phase0.save(
        result,
        BASELINE_DIR,
        config=config.as_kwargs(),
        package_versions=session.package_versions,
    )
    for name, path in paths.items():
        print(f"  saved {name}: {path}")

    paper_table = paper_style_table(
        result.per_scene_boa, result.per_scene_pa, result.per_scene_ua, result.pairable
    )
    paper_table_path = BASELINE_DIR / model_id / phase0.CLOUDSEN12_TEST / "paper_table.csv"
    paper_table.to_csv(paper_table_path)
    print(f"  saved paper_table: {paper_table_path}")

    results[model_version] = result.scores
    paper_tables[model_version] = paper_table

pd.concat(results, names=["model_version", "experiment"])

In [ ]:
pd.concat(paper_tables, names=["model_version", "experiment"])

In [ ]:
def version_label(model_version: float) -> str:
    return f"V{int(model_version)}"


median_tables = {version_label(v): table for v, table in paper_tables.items()}
comparison = compare(median_tables, column="median_boa")
colors = assign_colors(list(comparison.columns))
comparison

In [ ]:
figures = plot_experiment_comparison(
    comparison, colors, metric_label="Median BOA (%)"
)
for experiment, figure in figures.items():
    figure.savefig(
        BASELINE_DIR / f"ocm_versions_{experiment}_median_boa.png",
        dpi=150,
        bbox_inches="tight",
    )
    display(figure)

## Fase 2: OCM retreinado × Swin-T + UPerNet, com o OCM publicado como referência

In [ ]:
CACHE_DIR = Path("/content/cloudband_cache")
WORKERS = 32

fast = FastReader()
checks = ((test_table, 0), (test_table, len(test_table) - 1), (test_l2a_table, 0))
for table, position in checks:
    expected, got = gdal_read_sample(table, position), fast(table, position)
    same = np.array_equal(got.image, select_rgn(expected.image)) and np.array_equal(
        got.annotation, expected.annotation
    )
    assert same, f"the fast reader differs from GDAL at row {position}"
print(f"fast reader matches GDAL on {len(checks)} samples")

cache = LocalCache(CACHE_DIR)
TEST_SPLITS = (("test", test_table), ("test_l2a", test_l2a_table))
for split, table in TEST_SPLITS:
    cache.build(split, table, read_sample=fast, workers=WORKERS, bands_selected=True)

In [ ]:
PUBLISHED_VERSION = ocm.LATEST_MODEL_VERSION
PUBLISHED = "ocm published"
published_id = f"ocm-rgn-published-v{ocm.package_version()}-model{PUBLISHED_VERSION}"
published_config = ocm.InferenceConfig(
    patch_size=509, patch_overlap=0, model_version=PUBLISHED_VERSION
)

published = {}
for split, table in TEST_SPLITS:
    path = RESULTS_DIR / f"phase2-{PROTOCOL_REVISION}-counts-{split}-{published_id}.csv"
    if path.exists():
        published[split] = evaluation.load_counts(path)
        continue
    frame = evaluation.score_predictor(
        cache.samples(split, table),
        lambda stack: ocm.predict_array(stack, published_config),
        PUBLISHED,
    )
    evaluation.save_counts(frame, path)
    published[split] = frame
    print(f"{published_id}, {split}: scored and saved")

earlier = BASELINE_DIR / published_id / phase0.CLOUDSEN12_TEST / "per_scene_boa.csv"
if earlier.exists():
    gap = evaluation.legacy_check(
        published["test"], PUBLISHED, pd.read_csv(earlier, index_col=0)
    )
    print(f"largest gap to the per-patch BOA of the section above: {gap:.2e}")

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REGNETY, CONVNEXT = PAPER_BACKBONES


def seed_done(seed):
    names = [checkpoint_name(ocm_protocol, seed, backbone) for backbone in PAPER_BACKBONES]
    return all(map(store.is_done, names + [checkpoint_name(swin_protocol, seed)]))


def load_members(seed):
    def load(builder, name):
        return load_model_from_checkpoint(builder, store.checkpoint_path(name), DEVICE)

    return {
        "regnety": load(lambda: build_unet(REGNETY, pretrained=False),
                        checkpoint_name(ocm_protocol, seed, REGNETY)),
        "convnext": load(lambda: build_unet(CONVNEXT, pretrained=False),
                         checkpoint_name(ocm_protocol, seed, CONVNEXT)),
        "swin": load(lambda: build_swin_upernet(pretrained=False),
                     checkpoint_name(swin_protocol, seed)),
    }


seeds_ready = [s for s in ocm_protocol.seeds if seed_done(s)]
print("seeds finished for every model:", seeds_ready, "of", ocm_protocol.seeds)

counts = {split: {} for split, _ in TEST_SPLITS}
for seed in seeds_ready:
    members = None
    for split, table in TEST_SPLITS:
        path = RESULTS_DIR / f"phase2-{PROTOCOL_REVISION}-counts-{split}-seed{seed}.csv"
        if path.exists():
            counts[split][seed] = evaluation.load_counts(path)
            continue
        members = members or load_members(seed)
        frame = evaluation.score_models(
            cache.samples(split, table), members, {"ocm": ("regnety", "convnext")}, DEVICE
        )
        evaluation.save_counts(frame, path)
        counts[split][seed] = frame
        print(f"seed {seed}, {split}: scored and saved")
    del members
    gc.collect()
    torch.cuda.empty_cache()

for seed, frame in counts["test"].items():
    for model in ("ocm", "swin"):
        legacy = RESULTS_DIR / f"{model}_shared_per_scene_boa_seed{seed}.csv"
        if legacy.exists():
            gap = evaluation.legacy_check(frame, model, pd.read_csv(legacy, index_col=0))
            print(f"seed {seed} {model}: largest gap to the earlier per-patch BOA {gap:.2e}")

In [ ]:
if not counts["test"]:
    print("no seed is finished for every model yet")
else:
    tables = evaluation.report(
        evaluation.with_reference(counts["test"], published["test"]), published=PUBLISHED
    )
    print(f"L1C test (primary), seeds {sorted(counts['test'])}")
    for title, table in tables.items():
        print(f"\n{title}")
        display(table.round(4))

In [ ]:
# Same scenes at L2A: a robustness check, never pooled with the L1C result.
if not counts["test_l2a"]:
    print("no seed is finished for every model yet")
else:
    tables_l2a = evaluation.report(
        evaluation.with_reference(counts["test_l2a"], published["test_l2a"]), published=PUBLISHED
    )
    print(f"L2A test (secondary), seeds {sorted(counts['test_l2a'])}")
    for title, table in tables_l2a.items():
        print(f"\n{title}")
        display(table.round(4))
    primary_roi = evaluation.paired_comparison(counts["test"], "ocm", "swin", "roi")
    same = (
        np.sign(primary_roi["mean diff"]) == np.sign(tables_l2a["OCM vs Swin, per ROI"]["mean diff"])
    )
    print("\nOCM minus Swin per ROI has the same sign at L1C and L2A:", same.to_dict())

In [ ]:
def slug(title):
    return re.sub(r"[^a-z0-9]+", "-", title.lower()).strip("-")


saved = []
for split, name in (("test", "tables"), ("test_l2a", "tables_l2a")):
    for title, table in (globals().get(name) or {}).items():
        path = RESULTS_DIR / f"phase2-{PROTOCOL_REVISION}-report-{split}-{slug(title)}.csv"
        table.to_csv(path)
        saved.append(path.name)
print("saved to the shared results folder:" if saved else "nothing to save yet")
for name in saved:
    print("  ", name)

In [ ]:
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)
LEVELS = (("test", "L1C", "tables"), ("test_l2a", "L2A", "tables_l2a"))
COMPARISONS = (
    "OCM vs Swin, per ROI",
    "regnety alone vs Swin, per ROI",
    "convnext alone vs Swin, per ROI",
    f"OCM vs {PUBLISHED}, per ROI",
    f"Swin vs {PUBLISHED}, per ROI",
)

reports = {label: globals()[name] for _, label, name in LEVELS if globals().get(name)}
figures = {}
for split, label, name in LEVELS:
    if label in reports:
        figures[f"boa-per-seed-{split}"] = plot_seed_boa(
            reports[label]["whole split, per seed"], f"Whole-split BOA per seed, {label} test"
        )
if reports:
    figures["paired-differences-per-roi"] = plot_paired_differences(
        reports, COMPARISONS, "Paired differences per ROI, " + " and ".join(reports)
    )
for split, label, _ in LEVELS:
    if counts[split]:
        figures[f"ocm-minus-swin-per-roi-{split}"] = plot_unit_differences(
            evaluation.unit_differences(counts[split], "ocm", "swin", "roi"),
            "OCM", "Swin", f"OCM − Swin per ROI, {label} test",
        )

for name, figure in figures.items():
    figure.savefig(FIGURES_DIR / f"phase2-{PROTOCOL_REVISION}-{name}.png", dpi=200, bbox_inches="tight")
    display(figure)
print(f"{len(figures)} figures saved to {FIGURES_DIR}" if figures else "nothing to plot yet")